In [1]:
!py --version

Python 3.10.8


# import lib

In [2]:
!python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/                    
!python -m pip install "paddleocr[all]"                           
!pip install torch transformers                                      
!python -m pip install label-studio 


Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cpu/


# using paddleocr to get text and position of text
## save it in json file

In [ ]:
import json

from paddleocr import PaddleOCR

import os
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"


ocr = PaddleOCR(
    use_doc_orientation_classify=False, 
    use_doc_unwarping=False, 
    use_textline_orientation=False) # text detection + text recognition



image_path = "./training data/1.jpg"


result = ocr.predict(image_path) # predict the image
for res in result:
    res.print() # print the result of each image
    # res.save_to_img("output")
    res.save_to_json("output/paddle_ocr_result.json")



# json file ref label studio formate

# converting ocr output to label studio output 

In [ ]:
import json

def convert_ocr_to_ls(ocr_json, image_url, img_w, img_h):

    results = []

    polys = ocr_json["rec_polys"]
    texts = ocr_json["rec_texts"]

    for i, poly in enumerate(polys):

        xs = [p[0] for p in poly]
        ys = [p[1] for p in poly]

        x_min = min(xs)
        y_min = min(ys)

        x_max = max(xs)
        y_max = max(ys)

        w = x_max - x_min
        h = y_max - y_min

        results.append({
            "id": f"box_{i}",
            "type": "rectangle",
            "from_name": "bbox",
            "to_name": "image",
            "original_width": img_w,
            "original_height": img_h,
            "image_rotation": 0,
            "value": {
                "x": x_min/img_w*100,
                "y": y_min/img_h*100,
                "width": w/img_w*100,
                "height": h/img_h*100,
                "rotation": 0
            }
        })
        
        # OCR Text
        results.append({
            "id": f"box_{i}",
            "type": "textarea",
            "from_name": "transcription",
            "to_name": "image",
            "value": {
                "text": [texts[i]]
            }
        })
        
    return {
        "data": {
            "ocr": image_url
        },
        "predictions":[
            {
                "model_version":"paddleocr",
                "result": results
            }
        ]
    }

data=convert_ocr_to_ls(res, "http://localhost:8000/output/1.jpg", 2232, 3300)
print(data)
                       
# with open("./output/paddle_ocr_result.json", "r") as f:
#     data = json.load(f)
#     print(data.get('ocr'))
with open("output/labelstudio_tasks.json", "w") as f:
    json.dump(data, f, indent=4)


In [ ]:
import json
import pprint
with open("output/labelstudio_output.json", 'r') as f:
    data = json.load(f)
pprint.pprint(data)

In [ ]:
for task in data:
    annotations = task.get("annotations", [])
    print(annotation)

In [ ]:
import json


# Load your Label Studio JSON
with open("output/labelstudio_output.json", 'r') as f:
    data = json.load(f)

    

def normalize_bbox(x, y, width, height):
    """
    Convert % bbox → LayoutLM format (0–1000)
    """
    x1 = int(x * 10)
    y1 = int(y * 10)
    x2 = int((x + width) * 10)
    y2 = int((y + height) * 10)
    print("bbox")
    return [x1, y1, x2, y2]
    


def convert_label(label):
    """
    Convert labels to BIO format
    """
    print("label")
    if label.lower() == "ignore":
        return "O"
    return label.upper()



In [ ]:

final_dataset = []

for task in data:
    print(1)
    print(task["label"][0]["labels"][0])
    annotations = task["label"][0]["labels"][0]
    if not annotations:
        continue
    print(annotations)
    results = annotations["label"]

    words = []
    bboxes = []
    labels = []
    print(3)



    
    for item in results:
        print(4)
        if item["type"] != "labels":
            continue

        value = item["value"]

        # Extract data
        text = value.get("text", "")
        x = value.get("x", 0)
        y = value.get("y", 0)
        width = value.get("width", 0)
        height = value.get("height", 0)
        label = value.get("labels", ["O"])[0]

        # Split words
        tokens = text.split()

        # Convert bbox
        bbox = normalize_bbox(x, y, width, height)

        # Assign BIO labels
        for i, token in enumerate(tokens):
            words.append(token)
            bboxes.append(bbox)

            if label.lower() == "ignore":
                labels.append("O")
            else:
                if i == 0:
                    labels.append(f"B-{label.upper()}")
                else:
                    labels.append(f"I-{label.upper()}")

    final_dataset.append({
        "words": words,
        "bboxes": bboxes,
        "labels": labels
    })




In [ ]:
# Save cleaned dataset
with open("layoutlm_dataset.json", "w") as f:
    json.dump(final_dataset, f, indent=4)


print("✅ Dataset ready for LayoutLMv3 training!")

In [1]:
!pip install transformers datasets pillow torchvision


In [1]:
import torch
from transformers import LayoutLMv3Tokenizer, LayoutLMv3ForTokenClassification
from paddleocr import PaddleOCR
from PIL import Image
from torchvision import transforms


D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [3]:
! setx CUDA_HOME "C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v11.8"
! setx PATH "C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v11.8\bin;%PATH%"



SUCCESS: Specified value was saved.

SUCCESS: Specified value was saved.


In [12]:

# ---------------------------
# 1. Setup
# ---------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = LayoutLMv3Tokenizer.from_pretrained("microsoft/layoutlmv3-base")
model = LayoutLMv3ForTokenClassification.from_pretrained("microsoft/layoutlmv3-base")

import torch
print(torch.cuda.is_available())  # Should print True
print(torch.cuda.get_device_name(0))  # Shows GPU name

model.to(device)
model.eval()

ocr = PaddleOCR(use_gpu=True, lang='en')
# ocr = PaddleOCR(use_textline_orientation=True, lang='en')  # GPU if configured


Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                                | Status     | 
-----------------------------------+------------+-
layoutlmv3.embeddings.position_ids | UNEXPECTED | 
classifier.bias                    | MISSING    | 
classifier.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


True
NVIDIA GeForce RTX 4050 Laptop GPU


ValueError: Unknown argument: use_gpu

In [ ]:
# cheking is paddle ocr has gpu
!python -m pip install paddlepaddle-gpu
import paddle
gpu_available  = paddle.device.is_compiled_with_cuda()
print("GPU available:", gpu_available)

In [11]:
print(next(model.parameters()).device)
inputa = inputa.to(device)

print(inputa['input_ids'].device)
# !pip uninstall torch torchvision torchaudio


cuda:0


NameError: name 'inputs' is not defined

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121